# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [1]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

import os


load_dotenv(override=True)

# MODEL = "gpt-4.1-nano"
# MODEL = "openrouter/nex-agi/nex-n2.5-mini:free"
MODEL = "openai/unsloth/gemma-4-E4B-it-GGUF:Q4_K_M"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
API_KEY = ""

# BASE_URL = "https://openrouter.ai/api/v1"
BASE_URL = "http://127.0.0.1:8080/"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
# embedding_model = "all-MiniLM-L6-v2"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

openai = OpenAI(
    # api_key = "",
    # base_url = BASE_URL
)

In [2]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [3]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [4]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [5]:
documents = fetch_documents()

Loaded 76 documents


In [6]:
documents[0]

{'type': 'company',
 'source': 'knowledge-base/company/about.md',
 'text': "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a portfolio of 32 active cont

### Donezo! On to Step 2 - make the chunks

In [7]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [8]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# About Insurellm

Insurellm was founded by Avery La

In [9]:
documents[0]

{'type': 'company',
 'source': 'knowledge-base/company/about.md',
 'text': "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a portfolio of 32 active cont

In [10]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [11]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: company\nThe document has been retrieved from: knowledge-base/company/about.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 5 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n# A

In [12]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks, api_base=BASE_URL)
    reply = response.choices[0].message.content
    print(reply)
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [13]:
# def process_document(document):
#     messages = [
#         {
#             "role": "system",
#             "content": (
#                 "Return only valid JSON with this structure: "
#                 '{"chunks": [{"headline": "...", "summary": "...", "original_text": "..."}]}'
#             ),
#         },
#         {"role": "user", "content": make_prompt(document)},
#     ]
#     response = completion(model=MODEL, messages=messages, response_format=Chunks)
#     reply = response.choices[0].message.content
#     print(reply)
#     doc_as_chunks = Chunks.model_validate_json(reply).chunks
#     return [chunk.as_result(document) for chunk in doc_as_chunks]

In [14]:
process_document(documents[0])

{
  "chunks": [
    {
      "headline": "Founding and Initial Product Launch",
      "summary": "Insurellm was founded in 2015 by Avery Lancaster as an insurance tech startup, initially focusing on the Markellm marketplace before expanding into various insurance verticals.",
      "original_text": "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US."
    },
    {
      "headline": "Peak Growth and Strategic Restructuring",
      "summary": "After rapidly growing to 200 employees by 202

[Result(page_content='Founding and Initial Product Launch\n\nInsurellm was founded in 2015 by Avery Lancaster as an insurance tech startup, initially focusing on the Markellm marketplace before expanding into various insurance verticals.\n\n# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.', metadata={'source': 'knowledge-base/company/about.md', 'type': 'company'}),
 Result(page_content='Peak Growth and Strategic Restructuring\n\nAfter rapidly growing to 200 employees by 2020, Insure

In [15]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [16]:
chunks = create_chunks(documents)

  1%|▏         | 1/76 [00:28<35:55, 28.74s/it]

{
  "chunks": [
    {
      "headline": "Insurellm Founding and Early Product Development",
      "summary": "Insurellm was founded in 2015 by Avery Lancaster as an innovative insurance tech startup. Its initial product was Markellm, which connected consumers with insurance providers, laying the foundation for rapid expansion.",
      "original_text": "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US."
    },
    {
      "headline": "Growth Peak and Strategic Shift in Operations",
  

  3%|▎         | 2/76 [01:13<47:13, 38.29s/it]

{
  "chunks": [
    {
      "headline": "Insurellm: Company Overview and Mission",
      "summary": "Insurellm is a profitable, high-impact company founded in 2015 that has strategically restructured to maintain sustainable growth. They focus on delivering solutions through automation and AI for enterprise clients.",
      "original_text": "# Careers at Insurellm\n\n## Why Join Insurellm?\n\nAt Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.\n\nAfter reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic 

  4%|▍         | 3/76 [01:39<39:27, 32.43s/it]

...

{
  "chunks": [
    {
      "headline": "Lifellm Product Overview and Core Value Proposition",
      "summary": "Lifellm is an advanced life insurance platform by Insurellm that uses AI and predictive analytics to streamline underwriting, improve risk assessment, and enhance customer experiences for both B2B and B2C markets.",
      "original_text": "# Product Summary\n\n# Lifellm\n\n## Summary\n\nLifellm is an advanced life insurance platform developed by Insurellm that revolutionizes how insurance providers offer life coverage to their customers. Leveraging cutting-edge AI and predictive analytics, Lifellm enables insurers to provide personalized life insurance policies with streamlined underwriting, accurate risk assessment, and exceptional customer experiences. Designed for both B2B and B2C markets, Lifellm helps insurance companies modernize their life insurance operations while reducing costs and improving customer satisfaction."
    },
    {
      "headline": "AI Underwriting an

 99%|█████████▊| 75/76 [52:21<00:52, 52.45s/it]

{
  "chunks": [
    {
      "headline": "Markellm Product Overview: Bridging Consumers and Insurers",
      "summary": "Markellm is a two-sided marketplace using advanced AI to connect consumers seeking insurance with various insurance companies. It provides a seamless, personalized, and efficient platform for all parties involved in the complex insurance landscape.",
      "original_text": "# Product Summary\n\n# Markellm\n\n## Summary\n\nMarkellm is an innovative two-sided marketplace designed to seamlessly connect consumers with insurance companies. Powered by advanced matching AI, Markellm transforms the insurance shopping experience, making it more efficient, personalized, and accessible. Whether you're a homeowner searching for the best rates on home insurance or an insurer looking to reach new customers, Markellm acts as the ultimate bridge, delivering tailored solutions for all parties involved. With a user-friendly interface and powerful algorithms, Markellm not only saves tim

100%|██████████| 76/76 [53:07<00:00, 41.94s/it]

{
  "chunks": [
    {
      "headline": "Rellm: AI-Powered Enterprise Reinsurance Solution Overview",
      "summary": "Rellm is an innovative enterprise reinsurance product by Insurellm. It uses AI to revolutionize risk management, enhance decision-making, and improve operational efficiencies for reinsurance companies.",
      "original_text": "# Product Summary\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility."
    },
    {
      "headline": "Core Rellm Feat

In [17]:
print(len(chunks))

557


### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [18]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [20]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)

    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]

    # Generate local Hugging Face embeddings
    vectors = embeddings.embed_documents(texts)

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(
        ids=ids,
        embeddings=vectors,
        documents=texts,
        metadatas=metas,
    )

    print(f"Vectorstore created with {collection.count()} documents")

In [21]:
create_embeddings(chunks)

Vectorstore created with 557 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [22]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [23]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [24]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [25]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [26]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder, api_base=BASE_URL)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [27]:
# RETRIEVAL_K = 10

# def fetch_context_unranked(question):
#     query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
#     results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
#     chunks = []
#     for result in zip(results["documents"][0], results["metadatas"][0]):
#         chunks.append(Result(page_content=result[0], metadata=result[1]))
#     return chunks

In [28]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = embeddings.embed_query(question)
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [29]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [30]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

Annual Performa...
Professional Cr...
Alex Chen's Per...
Annual Performa...
Annual Performa...
Annual Performa...
Alex Chen's Per...
Professional Tr...
Career Progress...
Skills, Recogni...


In [31]:
reranked = rerank(question, chunks)

[8, 3, 10, 2, 5, 7, 4, 1, 9, 6]


In [32]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Professional Tr...
Alex Chen's Per...
Skills, Recogni...
Professional Cr...
Annual Performa...
Alex Chen's Per...
Annual Performa...
Annual Performa...
Career Progress...
Annual Performa...


In [33]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

17


In [34]:
reranked = rerank(question, chunks)

[18, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20]


In [35]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

0


In [36]:
reranked[0].page_content

"Professional Skills, Education, and Development\n\nOutlines Jessica Liu's educational background, including her BS in Computer Science. It details her technical proficiency in key areas like React and TypeScript, as well as her commitment to continuous learning and open-source contribution.\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester\n- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.\n- **Professional Development:** Completed Advanced React Patterns course (2023). Actively contributes to open-source projects."

In [37]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [38]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [39]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [40]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}], api_base=BASE_URL)
    return response.choices[0].message.content

In [41]:
rewrite_query("Who won the IIOTY award?", [])

'IIOTY award winner'

In [42]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages, api_base=BASE_URL)
    return response.choices[0].message.content, chunks

In [43]:
answer_question("Who won the IIOTY award?", [])

IIOTY award winner
[8, 12, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20]


('Maxine Thompson won the Insurellm Innovator of the Year (IIOTY) award in 2023.',
 [Result(page_content="Career Progression: Senior Data Engineer Status\n\nCovers Maxine's promotion to Senior Data Engineer, her contributions to improving data retrieval, and her recognition as Insurellm Innovator of the Year.\n\n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.", metadata={'source': 'knowledge-base/employees/Maxine Thompson.md', 'type': 'employees'}),
  Result(page_content="Other HR Notes and Future Development Goals\n\nOutlines Maxine's professional development, extracurricular activities (mentorship, wo

In [44]:
answer_question("Who went to Manchester University?", [])

Manchester University Insurellm
[8, 1, 3, 12, 18, 2, 17, 19, 5, 15, 14, 16, 7, 6, 9, 10, 11, 13, 4]


('I do not have any information in my knowledge base regarding who went to Manchester University.',
 [Result(page_content="Insurellm: Company Overview and Mission\n\nInsurellm is a profitable, high-impact company founded in 2015 that has strategically restructured to maintain sustainable growth. They focus on delivering solutions through automation and AI for enterprise clients.\n\n# Careers at Insurellm\n\n## Why Join Insurellm?\n\nAt Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.\n\nAfter reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized imp

In [45]:
answer_question("Tell me about the healthllm product", [])

healthllm product details
[3, 13, 4, 17, 10, 2, 8, 5, 12, 19, 1, 6, 7, 9, 11, 14, 15, 16, 18, 20]


("Hello! I'd be happy to tell you about Healthllm. As Insurellm's comprehensive health insurance platform, Healthllm is an AI-powered solution designed to help insurance providers modernize health coverage, improve outcomes for members, and navigate the complexities of the regulatory landscape.\n\nHere is a detailed overview of the product:\n\n### 🏥 Product Overview and Purpose\nHealthllm streamlines every aspect of health insurance operations, from initial plan design through claims processing and member engagement. Its primary goal is to empower insurers to deliver modern, personalized health coverage, helping them reduce costs and improve member outcomes.\n\n### ✨ Key Features\nHealthllm is equipped with several advanced capabilities:\n\n*   **Intelligent Plan Design:** The AI-powered tools analyze complex data—such as market trends, member demographics, and healthcare utilization patterns—to suggest optimal benefit structures, deductibles, and network configurations, maximizing bot